In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuration du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé pour l'entraînement : {device}")

Appareil utilisé pour l'entraînement : cpu


In [ ]:
# 1. Chargement et tri
df = pd.read_csv("../data/cleaned_dataset.csv", index_col=0)
df["delivery_time"] = pd.to_datetime(df["delivery_time"])
df = df.sort_values(["site_name", "delivery_time"]).reset_index(drop=True)

# 2. Filtrage des données aberrantes
df = df[df['to_drop_for_training'] != 1].copy()

# 3. Séparation des features et de la cible
EXCLUDE_COLS = [
    'site_name', 'delivery_time', 'production', 'installed_capacity', 
    'utilization_rate', 'is_maintenance', 'maint_rolling_sum', 
    'is_confirmed_maint', 'is_curtailment', 'to_drop_for_training'
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
TARGET_COL = 'utilization_rate'
SEQ_LEN = 24  # Séquence d'entrée (24h) et de sortie (24h)

print(f"Nombre de features utilisées : {len(FEATURE_COLS)}")

In [ ]:
def make_seq2seq_data(site_df, feature_cols, target_col, seq_len):
    values = site_df[feature_cols].values.astype(np.float32)
    target = site_df[target_col].values.astype(np.float32)
    times  = site_df['delivery_time'].values

    X, y, ts = [], [], []
    for i in range(len(site_df) - seq_len + 1):
        X.append(values[i : i + seq_len])
        y.append(target[i : i + seq_len])
        ts.append(times[i + seq_len - 1])

    return np.array(X), np.expand_dims(np.array(y), axis=-1), np.array(ts)

all_X, all_y, all_ts = [], [], []

for site, grp in df.groupby('site_name'):
    X_s, y_s, ts_s = make_seq2seq_data(grp, FEATURE_COLS, TARGET_COL, SEQ_LEN)
    all_X.append(X_s)
    all_y.append(y_s)
    all_ts.append(ts_s)

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)
all_ts = np.concatenate(all_ts, axis=0)

print(f"Shape globale X : {all_X.shape}") 
print(f"Shape globale y : {all_y.shape}")

In [ ]:
# 1. Tri chronologique global (Indispensable pour éviter le data leakage)
sort_idx = np.argsort(all_ts)
all_X_sorted = all_X[sort_idx]
all_y_sorted = all_y[sort_idx]
all_ts_sorted = all_ts[sort_idx]

# 2. Calcul des indices de coupure (70% Train, 15% Val, 15% Test)
n = len(all_y_sorted)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

# 3. Le découpage (Split)
X_train = all_X_sorted[:n_train]
y_train = all_y_sorted[:n_train]
ts_train = all_ts_sorted[:n_train]

X_val = all_X_sorted[n_train : n_train + n_val]
y_val = all_y_sorted[n_train : n_train + n_val]
ts_val = all_ts_sorted[n_train : n_train + n_val]

X_test = all_X_sorted[n_train + n_val :]
y_test = all_y_sorted[n_train + n_val :]
ts_test = all_ts_sorted[n_train + n_val :]

# 4. Vérification
print(f"Total des séquences : {n}\n")
print(f"Train : {X_train.shape[0]:>6} samples ({X_train.shape[0]/n*100:.1f}%) | {ts_train.min()} -> {ts_train.max()}")
print(f"Val   : {X_val.shape[0]:>6} samples ({X_val.shape[0]/n*100:.1f}%) | {ts_val.min()} -> {ts_val.max()}")
print(f"Test  : {X_test.shape[0]:>6} samples ({X_test.shape[0]/n*100:.1f}%) | {ts_test.min()} -> {ts_test.max()}")

In [ ]:
# 1. Normalisation (StandardScaler ajusté uniquement sur le Train)
scaler = StandardScaler()
n_train_s, seq_len_val, n_feat = X_train.shape

X_train_sc = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(n_train_s, seq_len_val, n_feat)
X_val_sc   = scaler.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)
X_test_sc  = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)

# 2. Conversion en Tenseurs PyTorch
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

X_val_t = torch.tensor(X_val_sc, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# 3. Création des DataLoaders
batch_size = 256
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)

In [ ]:
class WindSeq2SeqLSTM(nn.Module):
    def __init__(self, input_size, hidden_size1=128, hidden_size2=64, dropout=0.2):
        super(WindSeq2SeqLSTM, self).__init__()
        
        self.lstm1 = nn.LSTM(input_size, hidden_size1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        
        self.lstm2 = nn.LSTM(hidden_size1, hidden_size2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        
        self.fc1 = nn.Linear(hidden_size2, 32)
        self.relu = nn.ReLU()
        
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        out = self.dropout2(out)
        
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        
        return out

model = WindSeq2SeqLSTM(input_size=len(FEATURE_COLS)).to(device)
print(model)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 100
patience = 10
best_val_loss = float('inf')
counter = 0

for epoch in range(epochs):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        
        optimizer.zero_grad()
        preds = model(X_b)
        loss = criterion(preds, y_b)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * X_b.size(0)
    train_loss /= len(train_loader.dataset)
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model(X_b)
            loss = criterion(preds, y_b)
            val_loss += loss.item() * X_b.size(0)
    val_loss /= len(val_loader.dataset)
    
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # --- Early Stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_wind_lstm.pth')
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping déclenché à l'epoch {epoch+1}")
            break

# Chargement du meilleur modèle pour la suite
model.load_state_dict(torch.load('best_wind_lstm.pth'))

In [ ]:
# 1. Préparation et Inférence
X_test_t = torch.tensor(X_test_sc, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    y_test_pred_t = model(X_test_t)

y_test_pred = y_test_pred_t.cpu().numpy()

# 2. Métriques Globales
y_test_flat = y_test.flatten()
y_test_pred_flat = y_test_pred.flatten()

rmse = np.sqrt(mean_squared_error(y_test_flat, y_test_pred_flat))
mae = mean_absolute_error(y_test_flat, y_test_pred_flat)
r2 = r2_score(y_test_flat, y_test_pred_flat)

print("=== ÉVALUATION GLOBALE SUR LE TEST SET ===")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}\n")

# 3. Visualisation graphique d'une journée
idx = random.randint(0, len(y_test) - 1)
vraie_sequence = y_test[idx].flatten()
sequence_predite = y_test_pred[idx].flatten()
heures = np.arange(1, SEQ_LEN + 1)

plt.figure(figsize=(10, 4))
plt.plot(heures, vraie_sequence, label="Réel", color='blue', marker='o')
plt.plot(heures, sequence_predite, label="Prédiction LSTM", color='orange', marker='x', linestyle='--')

plt.title(f"Prévision sur 24h (Séquence n°{idx} du Test Set)")
plt.xlabel("Heure (de 1 à 24)")
plt.ylabel("Utilization Rate")
plt.ylim(0, 1.05)
plt.xticks(heures)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()